In [4]:
import pandas as pd
from sqlalchemy import create_engine
import requests
import time


engine = create_engine('sqlite:///terraria.db')


df = pd.read_excel('terraria-accessories.xlsx')

df.head()

,Name,Stage,Image
0,Aglet,pre-boss,NaN
1,Amber Horseshoe Balloon,pre-skeletron,NaN
2,Amphibian Boots,pre-hm,NaN
3,Anklet of the Wind,pre-boss,NaN
4,Arctic Diving Gear,pre-hm,NaN


In [5]:
print(df.shape)
df = df.drop_duplicates(subset=["Name"], keep="first")
print(df.shape)
df = df.dropna(subset=["Stage"])
print(df.shape)

(327, 3)
(305, 3)
(249, 3)


In [6]:
df['Stage'] = df['Stage'].map({
    "pre-boss" : "1 pre-boss",
    "pre-skeletron" : "2 pre-skeletron",
    "pre-hm" : "3 pre-hm",
    "pre-mech" : "4 pre-mech",
    "pre-plant" : "5 pre-plant",
    "pre-golem" : "6 pre-golem",
    "pre-lunar" : "7 pre-lunar",
    "post-moonlord" : "8 post-moonlord",
})

In [143]:
BASE_URL = 'https://terraria.wiki.gg/images/'

# Create the column if needed, preserving existing results
if 'Image' not in df.columns:
    df['Image'] = pd.Series(index=df.index, dtype='object')
else:
    df['Image'] = df['Image'].astype('object')

for index, row in df.iterrows():

    # Skip accessories already checked successfully
    if pd.notna(row['Image']):
        continue

    accessory = row['Name'].replace(" ", "_").replace("'","%27")
    

    url = BASE_URL + accessory + '.png'
    response = requests.get(url, timeout=15)
    print(url)
    print(response.status_code)

    # Respect rate limits and stop safely
    if response.status_code == 429:
        wait = response.headers.get('Retry-After', '60')
        print(f"Rate limited. Retry after {wait} seconds.")
        print(f"Stopped at row {index}. Rerun after the cooldown.")
        break

    if response.status_code == 200:
        df.at[index, 'Image'] = url

    elif response.status_code == 404:
        url = BASE_URL + accessory + '.gif'
        response = requests.get(url, timeout=15)

        if response.status_code == 429:
            wait = response.headers.get('Retry-After', '60')
            print(f"Rate limited. Retry after {wait} seconds.")
            print(f"Stopped at row {index}. Rerun after the cooldown.")
            break

        if response.status_code == 200:
            df.at[index, 'Image'] = url

    # Pause between accessories
    time.sleep(2)

print("Image lookup pass finished.")
print("Images found:", df['Image'].notna().sum())
print("Images remaining:", df['Image'].isna().sum())

Image lookup pass finished.
Images found: 249
Images remaining: 0


In [ ]:
# df.to_sql(
#     name='accessories',
#     con=engine,
#     if_exists='replace',
#     index=False
# )

249

In [7]:
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker
from sqlalchemy import delete, select, func

from db_models import *

engine = create_engine("sqlite:///./terraria.db")
Base.metadata.create_all(engine)
Session = sessionmaker(bind=engine)

with Session() as session:
    for _, row in df.iterrows():
        accessory = session.get(AccessoryDB, row["Name"])

        if accessory is not None:
            accessory.Stage = row["Stage"]

    session.commit()

with Session() as session:
    stmt = delete(AccessoryDB).where(
        AccessoryDB.Stage.is_(None)
    )

    result = session.execute(stmt)
    session.commit()


remove_extra_dev_wings = ["Will's Wings", "Crowno's Wings", "D-Town's Wings", "Cenx's Wings", "Lazure's Barrier Platform", "Yoraiz0r's Spell", "Loki's Wings", "Jim's Wings", "Skiphs' Paws", "Arkhalis' Lightwings", "Leinfors' Prehensile Cloak", "FoodBarbarian's Tattered Dragon Wings", "Grox The Great's Wings", "Safeman's Blanket Cape", "Ghostar's Infinity Eight", "Luna's Runic Pixie Wings", "Chippy's Cloak", "Heroicis' Wings", "Kazzymodus' Wings", "Chicken Bones' Wings"]

with Session() as session:
    count = session.scalar(
        select(func.count())
        .select_from(AccessoryDB)
        .where(AccessoryDB.Name.in_(remove_extra_dev_wings))
    )

    session.execute(
        delete(AccessoryDB).where(
            AccessoryDB.Name.in_(remove_extra_dev_wings)
        )
    )

    session.commit()

    print(f"Deleted {count} accessories.")

Deleted 0 accessories.
